In [2]:
# Install required libraries
!pip install plotly pandas numpy openpyxl kaleido -q

# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
from plotly.offline import init_notebook_mode
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Initialize Plotly for Colab
init_notebook_mode(connected=True)

# Load your retail sales data
df = pd.read_excel('/content/Retail_Sales_Data.xlsx')

# Display basic info about the dataset
print("📊 Dataset Overview:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nFirst few rows:")
display(df.head())

# Preprocess the data
def preprocess_retail_data(df):
    """Preprocess the retail sales data"""
    df_clean = df.copy()

    # Convert date column to datetime
    df_clean['Date'] = pd.to_datetime(df_clean['Date'])

    # Extract date features
    df_clean['Year'] = df_clean['Date'].dt.year
    df_clean['Month'] = df_clean['Date'].dt.month_name()
    df_clean['Day'] = df_clean['Date'].dt.day
    df_clean['Weekday'] = df_clean['Date'].dt.day_name()

    # Calculate sales amount and revenue
    df_clean['Sales_Amount'] = df_clean['Quantity'] * df_clean['Unit Price']
    df_clean['Revenue'] = df_clean['Sales_Amount'] * (1 - df_clean['Discount'])

    return df_clean

# Preprocess the data
df_processed = preprocess_retail_data(df)
print("✅ Data preprocessing completed!")

📊 Dataset Overview:
Shape: (50, 11)
Columns: ['Order ID', 'Date', 'Customer Name', 'Region', 'Product', 'Category', 'Quantity', 'Unit Price', 'Discount', 'Payment Method', 'Sales Rep']

First few rows:


,Order ID,Date,Customer Name,Region,Product,Category,Quantity,Unit Price,Discount,Payment Method,Sales Rep
0,1001,2024-03-22,Sunita Joshi,East,Keyboard,Electronics,4,2000,0.10,UPI,Alok Singh
1,1002,2024-01-15,Nikhil Roy,West,Keyboard,Electronics,5,2000,0.00,Debit Card,Alok Singh
2,1003,2024-01-04,Manish Yadav,South,Laptop,Electronics,3,25000,0.00,Credit Card,Pooja Mehta
3,1004,2024-02-05,Sunita Joshi,North,Office Table,Furniture,5,12000,0.15,Cash,Alok Singh
4,1005,2024-02-01,Priya Shah,West,Laptop,Electronics,5,2000,0.05,Credit Card,Alok Singh


✅ Data preprocessing completed!


In [3]:
class RetailSalesEDA:
    def __init__(self, df):
        self.df = df.copy()
        self.numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols = self.df.select_dtypes(include=['object']).columns.tolist()

    def dataset_overview(self):
        """Generate comprehensive dataset overview"""
        overview_data = {
            'Metric': ['Total Orders', 'Total Customers', 'Total Products',
                      'Total Regions', 'Date Range', 'Total Revenue',
                      'Total Sales Amount', 'Average Order Value'],
            'Value': [
                self.df.shape[0],
                self.df['Customer Name'].nunique(),
                self.df['Product'].nunique(),
                self.df['Region'].nunique(),
                f"{self.df['Date'].min().strftime('%Y-%m-%d')} to {self.df['Date'].max().strftime('%Y-%m-%d')}",
                f"₹{self.df['Revenue'].sum():,.2f}",
                f"₹{self.df['Sales_Amount'].sum():,.2f}",
                f"₹{self.df['Revenue'].mean():,.2f}"
            ]
        }

        fig = go.Figure(data=[go.Table(
            header=dict(
                values=['<b>Metric</b>', '<b>Value</b>'],
                fill_color='#2E86AB',
                align='left',
                font=dict(color='white', size=14)
            ),
            cells=dict(
                values=[overview_data['Metric'], overview_data['Value']],
                fill_color='#F5F5F5',
                align='left',
                font=dict(size=12)
            ))
        ])
        fig.update_layout(
            title='📊 Retail Sales Dataset Overview',
            title_x=0.5,
            height=400
        )
        return fig

    def sales_trend_analysis(self):
        """Analyze sales trends over time - FIXED VERSION"""
        # Monthly sales trend
        monthly_sales = self.df.groupby(self.df['Date'].dt.to_period('M')).agg({
            'Revenue': 'sum',
            'Order ID': 'count'
        }).reset_index()
        monthly_sales['Date'] = monthly_sales['Date'].dt.to_timestamp()

        # Create separate figures for better compatibility
        fig1 = go.Figure()
        fig1.add_trace(go.Scatter(
            x=monthly_sales['Date'],
            y=monthly_sales['Revenue'],
            mode='lines+markers',
            name='Revenue',
            line=dict(color='#2E86AB', width=3)
        ))
        fig1.update_layout(
            title='📈 Monthly Revenue Trend',
            xaxis_title='Month',
            yaxis_title='Revenue (₹)',
            height=400
        )

        fig2 = go.Figure()
        fig2.add_trace(go.Bar(
            x=monthly_sales['Date'],
            y=monthly_sales['Order ID'],
            name='Orders',
            marker_color='#A23B72'
        ))
        fig2.update_layout(
            title='📊 Monthly Order Count',
            xaxis_title='Month',
            yaxis_title='Number of Orders',
            height=400
        )

        return [fig1, fig2]

    def regional_analysis(self):
        """Analyze sales performance by region - FIXED VERSION"""
        regional_stats = self.df.groupby('Region').agg({
            'Revenue': ['sum', 'mean'],
            'Order ID': 'count',
            'Customer Name': 'nunique'
        }).round(2)
        regional_stats.columns = ['Total Revenue', 'Avg Revenue', 'Total Orders', 'Unique Customers']
        regional_stats = regional_stats.reset_index()

        # Create separate figures instead of subplots with incompatible types
        fig1 = go.Figure()
        fig1.add_trace(go.Pie(
            labels=regional_stats['Region'],
            values=regional_stats['Total Revenue'],
            hole=0.4,
            marker=dict(colors=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D'])
        ))
        fig1.update_layout(title='💰 Revenue Distribution by Region')

        fig2 = go.Figure()
        fig2.add_trace(go.Bar(
            x=regional_stats['Region'],
            y=regional_stats['Total Orders'],
            marker_color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
        ))
        fig2.update_layout(
            title='📦 Total Orders by Region',
            xaxis_title='Region',
            yaxis_title='Number of Orders'
        )

        fig3 = go.Figure()
        fig3.add_trace(go.Bar(
            x=regional_stats['Region'],
            y=regional_stats['Unique Customers'],
            marker_color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
        ))
        fig3.update_layout(
            title='👥 Unique Customers by Region',
            xaxis_title='Region',
            yaxis_title='Number of Customers'
        )

        fig4 = go.Figure()
        fig4.add_trace(go.Bar(
            x=regional_stats['Region'],
            y=regional_stats['Avg Revenue'],
            marker_color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
        ))
        fig4.update_layout(
            title='📊 Average Revenue per Order by Region',
            xaxis_title='Region',
            yaxis_title='Average Revenue (₹)'
        )

        return [fig1, fig2, fig3, fig4]

    def product_category_analysis(self):
        """Analyze product and category performance - FIXED VERSION"""
        category_stats = self.df.groupby('Category').agg({
            'Revenue': 'sum',
            'Order ID': 'count',
            'Quantity': 'sum'
        }).reset_index()

        product_stats = self.df.groupby('Product').agg({
            'Revenue': 'sum',
            'Order ID': 'count'
        }).nlargest(10, 'Revenue').reset_index()

        # Create separate figures
        fig1 = go.Figure()
        fig1.add_trace(go.Bar(
            x=category_stats['Category'],
            y=category_stats['Revenue'],
            marker_color=['#2E86AB', '#A23B72']
        ))
        fig1.update_layout(
            title='🏷️ Revenue by Category',
            xaxis_title='Category',
            yaxis_title='Revenue (₹)'
        )

        fig2 = go.Figure()
        fig2.add_trace(go.Bar(
            x=category_stats['Category'],
            y=category_stats['Order ID'],
            marker_color=['#F18F01', '#C73E1D']
        ))
        fig2.update_layout(
            title='📊 Orders by Category',
            xaxis_title='Category',
            yaxis_title='Number of Orders'
        )

        fig3 = go.Figure()
        fig3.add_trace(go.Bar(
            x=product_stats['Product'],
            y=product_stats['Revenue'],
            marker_color='#2E86AB'
        ))
        fig3.update_layout(
            title='🚀 Top 10 Products by Revenue',
            xaxis_title='Product',
            yaxis_title='Revenue (₹)',
            xaxis_tickangle=45
        )

        fig4 = go.Figure()
        fig4.add_trace(go.Bar(
            x=product_stats['Product'],
            y=product_stats['Order ID'],
            marker_color='#A23B72'
        ))
        fig4.update_layout(
            title='📦 Top 10 Products by Orders',
            xaxis_title='Product',
            yaxis_title='Number of Orders',
            xaxis_tickangle=45
        )

        return [fig1, fig2, fig3, fig4]

    def customer_analysis(self):
        """Analyze customer behavior and segmentation - FIXED VERSION"""
        customer_stats = self.df.groupby('Customer Name').agg({
            'Revenue': 'sum',
            'Order ID': 'count',
            'Quantity': 'sum'
        }).nlargest(10, 'Revenue').reset_index()

        # Customer segmentation by order frequency and revenue
        customer_segmentation = self.df.groupby('Customer Name').agg({
            'Order ID': 'count',
            'Revenue': 'sum'
        }).reset_index()

        # Create separate figures
        fig1 = go.Figure()
        fig1.add_trace(go.Bar(
            x=customer_stats['Customer Name'],
            y=customer_stats['Revenue'],
            marker_color='#2E86AB'
        ))
        fig1.update_layout(
            title='🏆 Top 10 Customers by Revenue',
            xaxis_title='Customer',
            yaxis_title='Revenue (₹)',
            xaxis_tickangle=45
        )

        fig2 = go.Figure()
        fig2.add_trace(go.Bar(
            x=customer_stats['Customer Name'],
            y=customer_stats['Order ID'],
            marker_color='#A23B72'
        ))
        fig2.update_layout(
            title='📊 Top 10 Customers by Orders',
            xaxis_title='Customer',
            yaxis_title='Number of Orders',
            xaxis_tickangle=45
        )

        fig3 = go.Figure()
        fig3.add_trace(go.Scatter(
            x=customer_segmentation['Order ID'],
            y=customer_segmentation['Revenue'],
            mode='markers',
            marker=dict(
                size=8,
                color=customer_segmentation['Revenue'],
                colorscale='Viridis',
                showscale=True
            ),
            text=customer_segmentation['Customer Name'],
            hovertemplate='<b>%{text}</b><br>Orders: %{x}<br>Revenue: ₹%{y:,.2f}<extra></extra>'
        ))
        fig3.update_layout(
            title='👑 Customer Segmentation',
            xaxis_title='Number of Orders',
            yaxis_title='Total Revenue (₹)'
        )

        fig4 = go.Figure()
        fig4.add_trace(go.Histogram(
            x=customer_segmentation['Revenue'],
            marker_color='#F18F01',
            nbinsx=20
        ))
        fig4.update_layout(
            title='📈 Revenue Distribution by Customer',
            xaxis_title='Revenue (₹)',
            yaxis_title='Number of Customers'
        )

        return [fig1, fig2, fig3, fig4]

    def payment_method_analysis(self):
        """Analyze payment method preferences - FIXED VERSION"""
        payment_stats = self.df.groupby('Payment Method').agg({
            'Revenue': 'sum',
            'Order ID': 'count',
            'Customer Name': 'nunique'
        }).reset_index()

        # Create separate figures
        fig1 = go.Figure()
        fig1.add_trace(go.Pie(
            labels=payment_stats['Payment Method'],
            values=payment_stats['Revenue'],
            hole=0.4,
            marker=dict(colors=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D'])
        ))
        fig1.update_layout(title='💳 Revenue Distribution by Payment Method')

        fig2 = go.Figure()
        fig2.add_trace(go.Bar(
            x=payment_stats['Payment Method'],
            y=payment_stats['Order ID'],
            marker_color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
        ))
        fig2.update_layout(
            title='📊 Orders by Payment Method',
            xaxis_title='Payment Method',
            yaxis_title='Number of Orders'
        )

        fig3 = go.Figure()
        fig3.add_trace(go.Bar(
            x=payment_stats['Payment Method'],
            y=payment_stats['Customer Name'],
            marker_color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
        ))
        fig3.update_layout(
            title='👥 Customers by Payment Method',
            xaxis_title='Payment Method',
            yaxis_title='Number of Customers'
        )

        return [fig1, fig2, fig3]

    def sales_rep_performance(self):
        """Analyze sales representative performance - FIXED VERSION"""
        rep_stats = self.df.groupby('Sales Rep').agg({
            'Revenue': 'sum',
            'Order ID': 'count',
            'Customer Name': 'nunique',
            'Quantity': 'sum'
        }).reset_index()

        # Create a single figure with compatible subplots
        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('💰 Revenue by Sales Rep', '📊 Orders by Sales Rep',
                          '👥 Customers by Sales Rep', '📦 Quantity Sold by Sales Rep'),
            vertical_spacing=0.15,
            horizontal_spacing=0.1
        )

        colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

        # Revenue
        fig.add_trace(
            go.Bar(x=rep_stats['Sales Rep'], y=rep_stats['Revenue'],
                   name="Revenue", marker_color=colors[0]),
            row=1, col=1
        )

        # Orders
        fig.add_trace(
            go.Bar(x=rep_stats['Sales Rep'], y=rep_stats['Order ID'],
                   name="Orders", marker_color=colors[1]),
            row=1, col=2
        )

        # Customers
        fig.add_trace(
            go.Bar(x=rep_stats['Sales Rep'], y=rep_stats['Customer Name'],
                   name="Customers", marker_color=colors[2]),
            row=2, col=1
        )

        # Quantity
        fig.add_trace(
            go.Bar(x=rep_stats['Sales Rep'], y=rep_stats['Quantity'],
                   name="Quantity", marker_color=colors[3]),
            row=2, col=2
        )

        fig.update_layout(
            height=600,
            showlegend=False,
            title_text="👨‍💼 Sales Representative Performance"
        )

        return fig

    def discount_analysis(self):
        """Analyze discount impact on sales - FIXED VERSION"""
        discount_impact = self.df.groupby('Discount').agg({
            'Revenue': 'mean',
            'Order ID': 'count',
            'Quantity': 'mean'
        }).reset_index()

        # Create separate figures
        fig1 = go.Figure()
        fig1.add_trace(go.Scatter(
            x=discount_impact['Discount'],
            y=discount_impact['Revenue'],
            mode='lines+markers',
            name="Avg Revenue",
            line=dict(color='#2E86AB', width=3)
        ))
        fig1.update_layout(
            title='📈 Average Revenue vs Discount',
            xaxis_title='Discount Rate',
            yaxis_title='Average Revenue (₹)'
        )

        fig2 = go.Figure()
        fig2.add_trace(go.Bar(
            x=discount_impact['Discount'],
            y=discount_impact['Order ID'],
            marker_color='#A23B72'
        ))
        fig2.update_layout(
            title='📊 Order Count vs Discount',
            xaxis_title='Discount Rate',
            yaxis_title='Number of Orders'
        )

        fig3 = go.Figure()
        fig3.add_trace(go.Scatter(
            x=discount_impact['Discount'],
            y=discount_impact['Quantity'],
            mode='lines+markers',
            name="Avg Quantity",
            line=dict(color='#F18F01', width=3)
        ))
        fig3.update_layout(
            title='📦 Average Quantity vs Discount',
            xaxis_title='Discount Rate',
            yaxis_title='Average Quantity'
        )

        return [fig1, fig2, fig3]

In [4]:
def generate_retail_sales_report(df):
    """
    Generate comprehensive retail sales EDA report - FIXED VERSION
    """
    eda = RetailSalesEDA(df)
    report_figures = {}

    print("🛍️  Generating Retail Sales EDA Report...")
    print("=" * 50)

    # 1. Dataset Overview
    print("📊 Generating dataset overview...")
    report_figures['overview'] = eda.dataset_overview()

    # 2. Sales Trend Analysis
    print("📈 Analyzing sales trends...")
    report_figures['sales_trends'] = eda.sales_trend_analysis()

    # 3. Regional Analysis
    print("🌍 Analyzing regional performance...")
    report_figures['regional_analysis'] = eda.regional_analysis()

    # 4. Product Category Analysis
    print("🏷️ Analyzing product categories...")
    report_figures['product_analysis'] = eda.product_category_analysis()

    # 5. Customer Analysis
    print("👥 Analyzing customer behavior...")
    report_figures['customer_analysis'] = eda.customer_analysis()

    # 6. Payment Method Analysis
    print("💳 Analyzing payment methods...")
    report_figures['payment_analysis'] = eda.payment_method_analysis()

    # 7. Sales Rep Performance
    print("👨‍💼 Analyzing sales representative performance...")
    report_figures['rep_performance'] = eda.sales_rep_performance()

    # 8. Discount Analysis
    print("🎯 Analyzing discount impact...")
    report_figures['discount_analysis'] = eda.discount_analysis()

    print("✅ Retail Sales EDA Report Complete!")
    print("=" * 50)

    return report_figures

# Generate the comprehensive report
print("Starting Automated EDA Report Generation...")
retail_report = generate_retail_sales_report(df_processed)

# Display all figures
print("\n🎨 Displaying EDA Report Visualizations...")
print("=" * 60)

for section, figures in retail_report.items():
    print(f"\n📊 Section: {section.replace('_', ' ').title()}")
    print("-" * 40)

    if isinstance(figures, list):
        for i, fig in enumerate(figures):
            fig.show()
    else:
        figures.show()

Starting Automated EDA Report Generation...
🛍️  Generating Retail Sales EDA Report...
📊 Generating dataset overview...
📈 Analyzing sales trends...
🌍 Analyzing regional performance...
🏷️ Analyzing product categories...
👥 Analyzing customer behavior...
💳 Analyzing payment methods...
👨‍💼 Analyzing sales representative performance...
🎯 Analyzing discount impact...
✅ Retail Sales EDA Report Complete!

🎨 Displaying EDA Report Visualizations...

📊 Section: Overview
----------------------------------------



📊 Section: Sales Trends
----------------------------------------



📊 Section: Regional Analysis
----------------------------------------



📊 Section: Product Analysis
----------------------------------------



📊 Section: Customer Analysis
----------------------------------------



📊 Section: Payment Analysis
----------------------------------------



📊 Section: Rep Performance
----------------------------------------



📊 Section: Discount Analysis
----------------------------------------


In [5]:
# Generate key business insights
def generate_business_insights(df):
    """Generate key business insights from the data"""

    print("💡 KEY BUSINESS INSIGHTS")
    print("=" * 50)

    # Top performing metrics
    top_product = df.groupby('Product')['Revenue'].sum().idxmax()
    top_region = df.groupby('Region')['Revenue'].sum().idxmax()
    top_customer = df.groupby('Customer Name')['Revenue'].sum().idxmax()
    top_sales_rep = df.groupby('Sales Rep')['Revenue'].sum().idxmax()
    most_used_payment = df.groupby('Payment Method')['Order ID'].count().idxmax()

    insights = {
        "🚀 Top Performing Product": top_product,
        "🏆 Top Performing Region": top_region,
        "⭐ Best Customer": top_customer,
        "👑 Top Sales Representative": top_sales_rep,
        "💳 Most Popular Payment Method": most_used_payment,
        "💰 Total Revenue": f"₹{df['Revenue'].sum():,.2f}",
        "📦 Total Orders": df['Order ID'].nunique(),
        "👥 Total Customers": df['Customer Name'].nunique(),
        "📈 Average Order Value": f"₹{df['Revenue'].mean():,.2f}",
        "🎯 Average Discount Rate": f"{df['Discount'].mean()*100:.1f}%"
    }

    for key, value in insights.items():
        print(f"{key}: {value}")

    print("\n📋 RECOMMENDATIONS:")
    print("• Focus on promoting top-performing products")
    print("• Expand successful regional strategies to other areas")
    print("• Implement loyalty programs for top customers")
    print("• Replicate strategies from top sales representatives")
    print("• Optimize discount strategies based on performance data")
    print("• Enhance payment options based on customer preferences")

# Generate insights
generate_business_insights(df_processed)

💡 KEY BUSINESS INSIGHTS
🚀 Top Performing Product: Office Table
🏆 Top Performing Region: West
⭐ Best Customer: Nikhil Roy
👑 Top Sales Representative: Alok Singh
💳 Most Popular Payment Method: Credit Card
💰 Total Revenue: ₹2,785,400.00
📦 Total Orders: 50
👥 Total Customers: 8
📈 Average Order Value: ₹55,708.00
🎯 Average Discount Rate: 6.1%

📋 RECOMMENDATIONS:
• Focus on promoting top-performing products
• Expand successful regional strategies to other areas
• Implement loyalty programs for top customers
• Replicate strategies from top sales representatives
• Optimize discount strategies based on performance data
• Enhance payment options based on customer preferences


In [6]:
# Function to export report as HTML
def export_report_to_html(report_figures, filename="retail_sales_eda_report.html"):
    """Export the complete EDA report as HTML"""
    with open(filename, 'w') as f:
        f.write('''
        <html>
        <head>
            <title>Retail Sales EDA Report</title>
            <style>
                body { font-family: Arial, sans-serif; margin: 20px; }
                .section { margin: 30px 0; padding: 20px; border: 1px solid #ddd; border-radius: 10px; }
                h1 { color: #2E86AB; text-align: center; }
                h2 { color: #A23B72; border-bottom: 2px solid #F18F01; padding-bottom: 10px; }
            </style>
        </head>
        <body>
            <h1>🛍️ Retail Sales Automated EDA Report</h1>
        ''')

        for section, figures in report_figures.items():
            section_title = section.replace('_', ' ').title()
            f.write(f'<div class="section"><h2>📊 {section_title}</h2>')

            if isinstance(figures, list):
                for i, fig in enumerate(figures):
                    f.write(f'<div style="margin: 20px 0;">')
                    f.write(fig.to_html(full_html=False, include_plotlyjs='cdn' if i == 0 else False))
                    f.write('</div>')
            else:
                f.write(figures.to_html(full_html=False, include_plotlyjs='cdn'))

            f.write('</div>')

        f.write('</body></html>')

    print(f"✅ Report exported as: {filename}")

# Export the report
export_report_to_html(retail_report)

✅ Report exported as: retail_sales_eda_report.html
